<a href="https://colab.research.google.com/github/Pawan-model/Huggingface-Experiment/blob/main/MRPC_Bert_FineTuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers
!pip install transformers[sentencepiece]
!pip install evaluate

In [ ]:
import torch
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import DataLoader
from datasets import load_dataset
from tqdm import tqdm




In [ ]:
checkpoint="bert-base-uncased"
tokenizer=AutoTokenizer.from_pretrained(checkpoint)
model=AutoModelForSequenceClassification.from_pretrained(checkpoint,num_labels=2)
raw_dataset=load_dataset('glue','mrpc')

In [ ]:
def tokenize_function(example):
  return tokenizer(example['sentence1'],example['sentence2'],truncation=True)
tokenized_dataset=raw_dataset.map(tokenize_function,batched=True)

In [ ]:
from transformers import DataCollatorWithPadding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

from transformers import TrainingArguments
training_args=TrainingArguments('test-trainer',eval_strategy='epoch')


In [ ]:
import numpy as np
from transformers import Trainer
import evaluate
def compute_metric(eval_preds):
  metric=evaluate.load('glue','mrpc')
  logits,labels=eval_preds
  predictions=np.argmax(logits,axis=-1)
  return metric.compute(predictions=predictions,references=labels)


In [ ]:
trainer=Trainer(model,
                training_args,
                train_dataset=tokenized_dataset['train'],
                eval_dataset=tokenized_dataset['validation'],
                data_collator=data_collator,
                processing_class=tokenizer,
                compute_metrics=compute_metric,)
trainer.train()